# Lab 04: Classification and Logistic Regression

Classification is a fundamental task in machine learning where the goal is to assign **discrete labels** to instances based on their inputs.
Suppose the label $y$ can take values in $\{ 0, 1 \}$.
This is known as **binary classification**.
Meanwhile, if $y$ can take more than two values, it is called **multiclass classification**.

We are going to implement a binary classifier using logistic regression for SMS spam detection.

## Data Loading and Preprocessing

We will use the [UCI SMS Spam Collection dataset](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) which contains a set of SMS messages labeled as "ham" (not spam) or "spam".

In [ ]:
import pandas as pd

df = pd.read_csv(
  "SMSSpamCollection",
  sep='\t', 
  header=None, 
  names=['label', 'message']
)

df

Now we convert the labels to binary values (0 for ham and 1 for spam):

In [ ]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df

Recall that logistic regression accepts an input $\mathbf{x} \in \mathbb{R}^d$.
Meanwhile, we have text data in the `message` column.
We need to convert the text data into a numerical format first. 

To do this, we will use `scikit-learn`'s `CountVectorizer` which converts a string into a vector of token/words counts.

Note: Run `%pip install scikit-learn` first if you don't have it installed.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=200)

# Equivalent to vectorizer.fit(df['message']) then vectorizer.transform(df['message'])
X = vectorizer.fit_transform(df["message"]).toarray()
y = df["label"].values

print(X.shape)  # (number of samples, number of features)
print(y.shape)  # (number of samples,)

## Logistic Regression Implementation

Recall that logistic regression models the probability of the positive class (spam) as:

$$
  p(y = 1 | \mathbf{x}) = \sigma(\boldsymbol{\theta}^\top \phi(\mathbf{x})) = \frac{1}{1 + \exp(-(\boldsymbol{\theta}^\top \phi(\mathbf{x}))} ,
$$

where $\phi(\mathbf{x})$ is the feature transformation of the input $\mathbf{x}$ (including the bias trick), and $\boldsymbol{\theta}$ are the parameters of the model.

So, first we implement the logistic-sigmoid function:

In [ ]:
import numpy as np

def logistic_sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))

Then, we can implement the logistic regression model:

In [ ]:
def predict_prob(X: np.ndarray, theta: np.ndarray) -> np.ndarray:
    X_hat = np.concatenate([X, np.ones([X.shape[0], 1])], axis=1)  # Bias trick
    z = X_hat @ theta
    return logistic_sigmoid(z)

We now need to define the loss function for logistic regression, which is the binary cross-entropy loss, or the negative Bernoulli log-likelihood:

$$
  \mathcal{L}(\boldsymbol{\theta}, b) = -\frac{1}{N} \sum_{n=1}^N \left[ y_n \log \sigma(\boldsymbol{\theta}^\top \phi(\mathbf{x}_n)) + (1 - y_n) \log (1 - \sigma(\boldsymbol{\theta}^\top \phi(\mathbf{x}_n))) \right]
$$

In [ ]:
def loss(X: np.ndarray, y: np.ndarray, theta: np.ndarray) -> float:
    p = predict_prob(X, theta)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

We also need to compute the gradient of the loss function with respect to the parameters $\boldsymbol{\theta}$:

$$
  \nabla_{\boldsymbol{\theta}} \mathcal{L} = -\frac{1}{N} \sum_{n=1}^N (y_n - \sigma(\boldsymbol{\theta}^\top \phi(\mathbf{x}_n))) \phi(\mathbf{x}_n) .
$$

In [ ]:
def gradient_loss(X: np.ndarray, y: np.ndarray, theta: np.ndarray) -> np.ndarray:
    p = predict_prob(X, theta)
    X_hat = np.concatenate([X, np.ones([X.shape[0], 1])], axis=1)  # Bias trick
    return -np.mean((y - p)[:, np.newaxis] * X_hat, axis=0)

Now we can implement the training loop using gradient descent:

In [ ]:
def train(
  X: np.ndarray,
  y: np.ndarray,
  theta: np.ndarray,
  learning_rate: float,
  num_iterations: int
) -> np.ndarray:
    for i in range(num_iterations):
        grad = gradient_loss(X, y, theta)
        theta -= learning_rate * grad

        if i % 100 == 0:
            current_loss = loss(X, y, theta)
            print(f"Iteration {i}, Loss: {current_loss:.4f}")

    return theta

First, we need to split the data into training and testing sets.
The test set will be used to evaluate the performance of our model after training.

In [ ]:
from sklearn.model_selection import train_test_split

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]}; test: {X_test.shape[0]}") 

Then we have the following training process:

In [ ]:
# Hyperparameters
learning_rate = 0.1
num_iterations = 1000

# Initialize parameters
theta = np.zeros(X.shape[1] + 1)  # +1 for bias term

# Train the model
theta = train(X_train, y_train, theta, learning_rate, num_iterations)

## Evaluation

Now can evaluate the classifier on the test set.
Recall that to obtain the label predictions, we can threshold the predicted probabilities at some $\tau$ (e.g., $\tau = 0.5$):

In [ ]:
p_pred = predict_prob(X_test, theta)

tau = 0.5
y_pred = (p_pred >= tau).astype(int)

Then, we compute the accuracy of our predictions:

In [ ]:
test_acc = np.mean(y_pred == y_test)
print(f"Test Accuracy: {test_acc:.4f} or {test_acc*100:.2f}%")

### Confusion Matrix

Which labels are being misclassified? To answer this question, we can compute the confusion matrix:

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

### Other Metrics

Recall that the elements of the confusion matrix are:

- True Positives (TP): Number of spam messages correctly classified as spam.
- True Negatives (TN): Number of ham messages correctly classified as ham.
- False Positives (FP): Number of ham messages incorrectly classified as spam.
- False Negatives (FN): Number of spam messages incorrectly classified as ham.

We can use `scikit-learn` to obtain them.
The function `ravel()` flattens the confusion matrix into a 1D array, by concatenating the rows.
Note that all these metrics depend on the choice of the threshold $\tau$.

Note: Run `%pip install seaborn` first if you don't have it installed.

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

plt.style.use("default")
plt.rcParams["font.family"] = ["Helvetica"]

import ipywidgets as widgets
from IPython.display import display

@widgets.interact(tau=widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01, description='τ'))
def sweep_threshold(tau):
    y_pred_tau = (p_pred >= tau).astype(int)
    cm_tau = confusion_matrix(y_test, y_pred_tau)
    tn, fp, fn, tp = cm_tau.ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    acc       = (tp + tn) / len(y_test)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))

    # Confusion matrix
    sns.heatmap(cm_tau, annot=True, fmt='d', cmap='Purples', ax=axes[0],
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    axes[0].set_title(f'Confusion Matrix (τ={tau:.2f})')

    # Metrics bar chart
    metrics = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}
    axes[1].bar(metrics.keys(), metrics.values(), color='mediumpurple')
    axes[1].set_ylim(0, 1)
    axes[1].set_title('Metrics')
    for i, v in enumerate(metrics.values()):
        axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10)

    plt.tight_layout()
    plt.show()

### Area Under the Curve (AUC)

Finally, we can also compute the Area Under the Receiver Operating Characteristic Curve (AUC-ROC) which is a common metric for binary classification.
It is defined as the area under the curve that plots the True Positive Rate (TPR) against the False Positive Rate (FPR) at various threshold settings.
Here's the plot of the ROC curve:

In [ ]:
from sklearn.metrics import roc_curve, auc

# fpr, tpr are for different threshold values stated in thresholds
fpr, tpr, thresholds = roc_curve(y_test, p_pred)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)
plt.title(f"AUROC = {roc_auc:.4f}")

plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("FPR")
plt.ylabel("TPR")

## Multi-Class Classification

So far we've dealt with binary labels $y \in \{0, 1\}$.
What if $y \in \{0, 1, ..., K-1\}$? 
The Bernoulli likelihood generalizes to the Categorical likelihood, and logistic-sigmoid generalizes to softmax.
The Categorical distribution is a special case of the Multinomial distribution.
Often, the term "softmax regression" or "multinomial logistic regression" is used to refer to the generalization of logistic regression to the multiclass setting.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

# Generate a synthetic 3-class dataset in 2D
X_multi, y_multi = make_blobs(n_samples=600, centers=3, cluster_std=1.5, random_state=42)

X_multi_train, X_multi_test, y_multi_train, y_multi_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

plt.scatter(X_multi[:, 0], X_multi[:, 1], c=y_multi, cmap='viridis', edgecolors='k', s=20)

Let's train a softmax regression model on this dataset.

In [ ]:
# Fit softmax regression. 
# Scikit-learn automatically detects multiclass data, then
# uses the correct Categorical likelihood with softmax.
clf = LogisticRegression()
clf.fit(X_multi_train, y_multi_train)

And now we can evaluate the model on the test set.

In [ ]:
def plot_decision_regions(X, y, clf, ax, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='viridis')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='k', s=20)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_decision_regions(X_multi_train, y_multi_train, clf, axes[0], 'Decision Regions (Train)')
plot_decision_regions(X_multi_test,  y_multi_test,  clf, axes[1], 'Decision Regions (Test)')

plt.tight_layout()
plt.show()